# Displaying a numpy field in McIDAS-V

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## Build an analytic field on a lat/lon grid

In [ ]:
import numpy as np
import mcidasv_jupyter as mcv
session = mcv.get_session()

lats = np.linspace(20, 52, 200)
lons = np.linspace(-125, -66, 360)
LON, LAT = np.meshgrid(lons, lats)

def blob(lat0, lon0, amp, sigma):
    return amp * np.exp(-(((LAT-lat0)**2 + (LON-lon0)**2) / (2*sigma**2)))

field = (280
         + blob(40, -100, 18, 6.0)
         - blob(33, -85, 15, 5.0)
         + blob(45, -75, 8, 4.0)).astype('f4')
print('field shape', field.shape, 'range', float(field.min()), float(field.max()))

## Write it to a grid and display it

In [ ]:
session.run('''
panel = buildWindow(height=600, width=800)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setEnhancement('Temperature', range=(262, 298))
''', arrays={'g': (field, lats, lons)})

## Add metadata (units, long name)

In [ ]:
grid = session.write_grid(field, lats, lons, name='tair',
                          attrs={'units': 'K', 'long_name': 'synthetic air temperature'})
print('wrote', grid.path, '(field =', grid.field + ')')

session.run('''
panel = buildWindow(height=600, width=800)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
layer.setLayerLabel(label='synthetic T (K) from numpy')
''', arrays={'g': grid})